背景
神经网络（NN）是作用于输入 数据的一系列嵌套函数集合。这些函数由参数（包含权重和偏置）定义，在 PyTorch 中这些参数存储在张量（tensors）中。数据管理

神经网络的训练分为两步

前向传播（Forward Propagation）：在前向传播中，神经网络对正确的输出做出最佳猜测。它将输入数据传入每一层函数以得出此猜测。

反向传播（Backward Propagation）：在反向传播中，神经网络根据其猜测的误差比例调整参数。它通过从输出端向后遍历，收集误差相对于函数参数的导数（梯度），并使用梯度下降优化参数。关于反向传播的更详细讲解，请查看这个 3Blue1Brown 视频。

PyTorch 中的使用方法
让我们看一个单一的训练步骤。在此示例中，我们从 torchvision 加载一个预训练的 resnet18 模型。我们创建一个随机数据张量来代表单个图像（3 通道，高宽均为 64），并将其对应的 label 初始化为随机值。预训练模型中的标签形状为 (1,1000)。

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\86136/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


前向传播

In [2]:
prediction = model(data) # forward pass

我们使用模型的预测值和对应的标签来计算误差（loss）。下一步是将该误差通过网络进行反向传播。当我们对误差张量调用 .backward() 时，反向传播正式启动。此时 Autograd 会计算每个模型参数的梯度，并将其存储在参数的 .grad 属性中。

In [3]:
loss = (prediction - labels).sum()
loss.backward() # backward pass

接下来，我们加载一个优化器，本例中使用学习率为 0.01 且动量（momentum）为 0.9 的 SGD。我们将模型的所有参数注册到优化器中。

In [4]:
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

最后，我们调用 .step() 来启动梯度下降。优化器会根据存储在 .grad 中的梯度来调整每个参数。

In [5]:
optim.step() #gradient descent

至此，你已经具备了训练神经网络所需的一切。以下章节详细介绍了 autograd 的工作原理——你可以跳过它们。

# Autograd 中的微分
让我们看看 autograd 是如何收集梯度的。我们创建两个 requires_grad=True 的张量 a 和 b。这向 autograd 发出信号，表明应对它们执行的所有操作进行跟踪。

In [6]:
import torch

a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

我们通过 a 和 b 创建另一个张量 Q。

In [7]:
Q = 3*a**3 - b**2

假设 a 和 b 是神经网络的参数，而 Q 是误差。在神经网络训练中，我们需要误差相对于参数的梯度，
当我们对 Q 调用 .backward() 时，autograd 会计算这些梯度并将其存储在各自张量的 .grad 属性中。物理学
由于 Q 是一个向量，我们需要在 Q.backward() 中显式传递一个 gradient 参数。gradient 是一个与 Q 形状相同的张量，它代表 Q 相对于其自身的梯度，
同样地，我们也可以将 Q 聚合为一个标量，然后隐式调用 backward，例如 Q.sum().backward()。

In [8]:
external_grad = torch.tensor([1., 1.])
Q.backward(gradient=external_grad)

梯度现在已存入 a.grad 和 b.grad 中。

In [9]:
# check if collected gradients are correct
print(9*a**2 == a.grad)
print(-2*b == b.grad)

tensor([True, True])
tensor([True, True])


# 计算图（Computational Graph）
从概念上讲，autograd 在一个有向无环图（DAG）中记录了 数据（张量）及所有已执行的操作（以及由此产生的新张量），该图由 Function 对象组成。在这个 DAG 中，叶子节点是输入张量，根节点是输出张量。通过从根节点回溯到叶子节点，你可以利用链式法则自动计算梯度。

在前向传播中，autograd 同时执行两项操作：

运行请求的操作以计算结果张量，以及

在 DAG 中维护该操作的梯度函数。

当对 DAG 根节点调用 .backward() 时，反向传播开始。此时 autograd 会：

根据每个 .grad_fn 计算梯度，

将它们累加到对应张量的 .grad 属性中，并且

利用链式法则，一直传播到叶子张量。

下方是我们示例中 DAG 的可视化表示。图中箭头方向为前向传播的方向。节点代表前向传播中每个操作的逆向函数。蓝色的叶子节点代表我们的叶子张量 a 和 b。



# 从 DAG 中排除
torch.autograd 会跟踪所有 requires_grad 标志设置为 True 的张量上的操作。对于不需要梯度的张量，将此属性设置为 False 可将其排除在梯度计算 DAG 之外。

即使只有一个输入张量的 requires_grad=True，操作的输出张量也需要梯度。

In [10]:
x = torch.rand(5, 5)
y = torch.rand(5, 5)
z = torch.rand((5, 5), requires_grad=True)

a = x + y
print(f"Does `a` require gradients?: {a.requires_grad}")
b = x + z
print(f"Does `b` require gradients?: {b.requires_grad}")

Does `a` require gradients?: False
Does `b` require gradients?: True


在神经网络中，不计算梯度的参数通常被称为冻结参数。如果你预先知道不需要某些参数的梯度，那么“冻结”模型的一部分非常有用（这可以通过减少 autograd 计算提供一些性能优势）。

在微调（finetuning）中，我们会冻结大部分模型，通常只修改分类器层以对新标签进行预测。让我们通过一个小例子来演示。和之前一样，我们加载预训练的 resnet18 模型，并冻结所有参数。

In [11]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all the parameters in the network
for param in model.parameters():
    param.requires_grad = False

假设我们想在包含 10 个标签的新 数据集上微调模型。在 resnet 中，分类器是最后一层线性层 model.fc。我们可以简单地将其替换为一个新的线性层（默认未冻结），作为我们的分类器。

In [12]:
model.fc = nn.Linear(512, 10)

现在模型中的所有参数（除 model.fc 的参数外）都被冻结了。唯一计算梯度的参数是 model.fc 的权重和偏置。

In [13]:
# Optimize only the classifier
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

请注意，尽管我们将所有参数注册到优化器中，但唯一计算梯度（并在梯度下降中更新）的参数是分类器的权重和偏置。

同样的排除功能可以通过上下文管理器 torch.no_grad() 使用。